In [244]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, Conv2DTranspose, BatchNormalization, Dropout, Activation, LayerNormalization, MultiHeadAttention, Add, Dense, Concatenate, ReLU
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import load_model

In [245]:
from tensorflow.keras.layers import GlobalAveragePooling2D, Reshape

In [246]:
# from keras import backend as K
from tensorflow.keras import layers, models, backend as K

In [247]:
# atrous convolutional layer architecture
def atrous_conv_layer(x, features, kernel_size=(3,3), dilation_rate = 1):

    # padding = dilation_rate if kernel_size == (3,3) else 0
    atrous_conv = Conv2D(features, kernel_size, 
                                padding='same', dilation_rate=dilation_rate)(x)
    batch_norm = BatchNormalization()(atrous_conv)
    relu_actv = ReLU()(batch_norm)

    # returns a layer (structure)
    return relu_actv

# global pooling architecture
def global_pooling_layer(x, features):

    image_pooling = GlobalAveragePooling2D()(x)
    image_pooling = Reshape((1, 1, K.int_shape(x)[-1]))(image_pooling)
    print("image global pooling should be (1,1,1024): ", image_pooling)
    conv1x1 = atrous_conv_layer(image_pooling, features, kernel_size=1)
    global_avg_pool = UpSampling2D(size=(K.int_shape(x)[1], K.int_shape(x)[2]), interpolation="bilinear")(conv1x1)
    
    return global_avg_pool

In [248]:
# Atrous Spatial Pyramid pool function ---> parallel convolution + global pooling + 1x1 conv + concat 
def ASPP_multiple_conv(x, features):
    print("ASPP")
    print("features: ", features)

    # 1x1 convolutions with dilation rate = 1
    branch1 = atrous_conv_layer(x, features, kernel_size=1)
    print("branch1: ", branch1)
    # 3x3 convolutions with different dilation rates
    branch2 = atrous_conv_layer(x, features, kernel_size=3, dilation_rate=6)
    print("branch2: ", branch2)
    branch3 = atrous_conv_layer(x, features, kernel_size=3, dilation_rate=12)
    print("branch3: ", branch3)
    branch4 = atrous_conv_layer(x, features, kernel_size=3, dilation_rate=18)
    print("branch4: ", branch4)

    global_pooling = global_pooling_layer(x, features)

    print("global pooling: ", global_pooling)
    # concat + 1x1 conv is done here
    concat_output  = concatenate([branch1, branch2, branch3, branch4, global_pooling])
    print("concat output: ", concat_output)
    aspp_output = atrous_conv_layer(concat_output, features, kernel_size=1)
    print("aspp_output: ", aspp_output)
    return aspp_output

In [249]:
def decoder(x, features, num_classes, input_shape):

    x = UpSampling2D(size=(4, 4), interpolation="bilinear")(x)  # Adjust size to match input shape
    x = atrous_conv_layer(x, 512, kernel_size=3)

    x = UpSampling2D(size=(4, 4), interpolation="bilinear")(x)
    x = atrous_conv_layer(x, 256, kernel_size=3)
   
    # x = Conv2D(num_classes, kernel_size=1, padding="same")(x)  # Output layer for segmentation
    # print("------------------")
    # print("working right till here")
    # print("------------------")
    # print(x)
    
    # Bilinear upsampling to match input shape
    # x = UpSampling2D(size=(input_shape[0] // K.int_shape(x)[1], input_shape[1] // K.int_shape(x)[2]), interpolation="bilinear")(x)

    # x = atrous_conv_layer(x, 25, kernel_size=1)

    output = Conv2D(25, 1, padding="same", activation="softmax")(x)
    
    return output

In [250]:
def deeplabv3(deeplab_input, backbone_output, num_classes, features):

    # inputs = Input(shape=deeplab_input)
    # print(inputs)
    aspp_output = ASPP_multiple_conv(backbone_output, features=features)
    upsampled_decoder_output = decoder(aspp_output, features, num_classes, deeplab_input)
    
    # Create model
    return models.Model(deeplab_input, upsampled_decoder_output, name="DeepLabV3")

In [251]:
# backbone_input_shape = (256, 256, 3)
# backbone_inputs = Input(shape=backbone_input_shape)
input_shape = Input(shape=(256, 256, 3))

In [252]:
base_model = tf.keras.applications.ResNet50(weights="imagenet", include_top=False, input_tensor=input_shape)
backbone_output = base_model.get_layer("conv4_block6_out").output

In [253]:
# deeplab_input = backbone_output.shape[1:]
# deeplab_input

In [254]:
model = deeplabv3(input_shape, backbone_output, 25, 1024)

ASPP
features:  1024
branch1:  <KerasTensor shape=(None, 16, 16, 1024), dtype=float32, sparse=False, name=keras_tensor_4930>
branch2:  <KerasTensor shape=(None, 16, 16, 1024), dtype=float32, sparse=False, name=keras_tensor_4933>
branch3:  <KerasTensor shape=(None, 16, 16, 1024), dtype=float32, sparse=False, name=keras_tensor_4936>
branch4:  <KerasTensor shape=(None, 16, 16, 1024), dtype=float32, sparse=False, name=keras_tensor_4939>
image global pooling should be (1,1,1024):  <KerasTensor shape=(None, 1, 1, 1024), dtype=float32, sparse=False, name=keras_tensor_4941>
global pooling:  <KerasTensor shape=(None, 16, 16, 1024), dtype=float32, sparse=False, name=keras_tensor_4945>
concat output:  <KerasTensor shape=(None, 16, 16, 5120), dtype=float32, sparse=False, name=keras_tensor_4946>
aspp_output:  <KerasTensor shape=(None, 16, 16, 1024), dtype=float32, sparse=False, name=keras_tensor_4949>


In [255]:
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [256]:
model.summary()

Model: "DeepLabV3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_54      │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 262, 262,  │          0 │ input_layer_54[0… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 128, 128,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 128, 128,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 128, 128,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 130, 130,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 64, 64,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 64, 64,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 64, 64,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 64, 64,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 64, 64,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 64, 64,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 64, 64,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 64, 64,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 64, 64,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 64, 64,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 64, 64,    │      1,024 │ conv2_block1_3_c

 Total params: 50,179,993 (191.42 MB)

 Trainable params: 50,135,577 (191.25 MB)

 Non-trainable params: 44,416 (173.50 KB)

In [257]:
# model.fit(
#     data_generator(image_dir, mask_dir, batch_size),
#     epochs=epochs,
#     steps_per_epoch=steps_per_epoch
# )